<a href="https://colab.research.google.com/github/mu-janane-27/Innomatics-Research-lab-tasks/blob/main/Task%203%20Feb%20Internship%20NLP%20Build%20a%20Chatbot%20using%20Hugging%20Face%20Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 1: Model Loading

In [1]:
!pip install transformers torch

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "microsoft/DialoGPT-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/641 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/351M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-small
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Task 2: User Input Handling

The chatbot takes input from the user using a loop.  
It ensures continuous interaction until the user decides to exit.
 (Handled inside main loop)

Task 3: Response Generation

The chatbot generates responses using the transformer model.  
User input is converted into tokens and passed to the model to generate meaningful replies.

Task 4: Continuous Conversation

The chatbot maintains conversation context by storing previous chat history.  
This allows it to generate context-aware responses.

In [4]:
chat_history_ids = None

print("Chatbot: Hello! I am your AI assistant. How can I help you today?")

while True:
    user_input = input("You: ")

    # Exit condition
    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a great day 😊")
        break

    # Encode input
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    # Maintain chat history
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=1000,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.75,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode response
    bot_output = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    print("Chatbot:", bot_output)

Chatbot: Hello! I am your AI assistant. How can I help you today?
You: Hello
Chatbot: Oh my God I'm laughing so hard.
You: What is Artificial Intelligence?
Chatbot: I'm going to say this but I'm pretty sure it's AI, not Artificial Intelligence.
You: exit
Chatbot: Goodbye! Have a great day 😊


Task 5: Exit Condition

The chatbot stops running when the user types "exit" or "quit".  
This is handled using a conditional statement inside the loop.